# CHC (CHIRPS) — catalog explorer (no network)

The `chc` backend reaches the full Climate Hazards Center product line — CHIRPS / CHIRP precipitation, CHIRTS temperature & humidity, CHIRPS-GEFS forecasts, CHPclim climatology, WBGT, and the SPI / SPEI drought indices — over anonymous FTP. This notebook explores the **bundled catalog offline**: no FTP access, so it runs deterministically at docs-build time.

In [1]:
from earthlens.chc import Catalog

cat = Catalog()
print(f'datasets in catalog: {len(cat.datasets)}')
print(f'available_datasets index: {len(cat.available_datasets)}')
print(f'named regions: {len(cat.available_regions)}')

2026-06-07 17:41:30 | INFO | pyramids.base.config | Logging is configured.


datasets in catalog: 97
available_datasets index: 97
named regions: 9


## Datasets by temporal resolution

Each catalog entry knows its native cadence (daily, monthly, dekadal, pentadal, …) and the pandas frequency the backend builds its date axis from.

In [2]:
from collections import Counter

by_res = Counter(d.temporal_resolution for d in cat.datasets.values())
for res, n in sorted(by_res.items(), key=lambda kv: -kv[1]):
    print(f'{n:>3}  {res}')

 19  monthly
 19  pentadal
 16  daily
 16  daily-delta
 12  dekadal
  4  2-monthly
  3  3-monthly
  2  annual
  1  6-hourly
  1  monthly-climatology
  1  seasonal
  1  5-day
  1  10-day
  1  15-day


## Inspect one dataset

`global-daily` is the headline CHIRPS-2.0 product — 0.05° global daily rainfall back to 1981.

In [3]:
d = cat.datasets['global-daily']
print('region            :', d.region)
print(
    'temporal_resolution:', d.temporal_resolution, '(pandas freq', d.pandas_freq + ')'
)
print('spatial_resolution :', d.spatial_resolution, 'deg')
print('lat boundaries     :', d.lat_boundaries)
print('lon boundaries     :', d.lon_boundaries)
print('start_date         :', d.start_date)
print('formats            :', d.formats)
print('variables          :', list(d.variables))

region            : global
temporal_resolution: daily (pandas freq D)
spatial_resolution : [0.05] deg
lat boundaries     : [-50.0, 50.0]
lon boundaries     : [-180.0, 180.0]
start_date         : 1981-01-01
formats            : ['tif']
variables          : ['precipitation']


## Look up a variable

Every dataset exposes a `variables` map of `Variable` metadata — name, units (via type), and a human description.

In [4]:
var = cat.datasets['global-daily'].variables['precipitation']
print('name       :', var.name)
print('type        :', var.types)
print('description :', var.description)

name       : precipitation
type        : flux
description : Daily rainfall estimate


## Named regions

CHC ships several regional sub-grids; the catalog expands per-region dataset variants from this block.

In [5]:
for name, box in cat.available_regions.items():
    print(f'{name:<28} {box}')

global                       {'lat_boundaries': [-50.0, 50.0], 'lon_boundaries': [-180.0, 180.0]}
global-land                  {'lat_boundaries': [-60.0, 70.0], 'lon_boundaries': [-180.0, 180.0]}
global-extended              {'lat_boundaries': [-90.0, 90.0], 'lon_boundaries': [-180.0, 180.0]}
africa                       {'lat_boundaries': [-40.0, 40.0], 'lon_boundaries': [-20.0, 55.0]}
central-america-caribbean    {'lat_boundaries': [5.0, 35.0], 'lon_boundaries': [-120.0, -55.0]}
east-africa                  {'lat_boundaries': [-12.0, 6.0], 'lon_boundaries': [28.0, 42.0]}
east-africa-centennial       {'lat_boundaries': [-12.25, 22.25], 'lon_boundaries': [21.25, 51.25]}
indonesia                    {'lat_boundaries': [-11.0, 6.0], 'lon_boundaries': [95.0, 141.0]}
western-hemisphere           {'lat_boundaries': [-50.0, 50.0], 'lon_boundaries': [-180.0, 0.0]}
